<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-10-production-deployment/lesson-10.3-ai-gateway-pattern/notebooks/exercises-10.3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10.3 — AI Gateway Pattern
Netsetos GenAI Course v2.0 | Module 10: Production Deployment

Unified API for 100+ providers, fallback, virtual keys, semantic cache, cost routing, observability, HA.


In [ ]:
print('Module 10: Production Deployment')
print('Lesson 10.3: AI Gateway Pattern')


## Ex 1: What a Gateway Solves


In [ ]:
print('Without gateway: N providers x M services = chaos')
for problem, gateway_solution in [
    ('Auth per provider per service','Centralized virtual keys'),
    ('Rate-limit plumbing duplicated','One middleware'),
    ('Provider outage = app outage','Automatic fallback'),
    ('Cost attribution broken','Per-tenant metadata tagging'),
    ('Observability fragmented','One hook for Langfuse/Helicone'),
    ('Vendor lock-in','OpenAI-compatible shim'),
]:
    print(f'  {problem:36s} -> {gateway_solution}')


## Ex 2: Gateway Tools Compared


In [ ]:
print('Gateway tool landscape:')
for tool, strength, best_for in [
    ('LiteLLM','Open source, 100+ providers, OpenAI-compatible','Self-host default'),
    ('Portkey','SaaS, 400B+ tok/day, polished UI','Enterprise w/o ops team'),
    ('Bifrost','Rust, 11us overhead per call','Latency-critical paths'),
    ('Helicone','Drop-in proxy + fast analytics','Quick cost visibility'),
    ('Cloudflare AI Gateway','Edge-caching LLM responses','Geo-distributed apps'),
]:
    print(f'  {tool:22s}: {strength:48s} | Best: {best_for}')


## Ex 3: Fallback Priority List


In [ ]:
print('Fallback strategies:')
for strat, trigger, latency_cost in [
    ('Static priority list','5xx/429/timeout','+1 round-trip on failure'),
    ('Per-provider breaker','5 fails / 60s window','Quarantines bad provider'),
    ('Quality-based','Eval score < threshold','Needs LLM judge in path'),
    ('Region-based','Latency > p99 SLO','Geo failover, GDPR friendly'),
]:
    print(f'  {strat:22s}: {trigger:30s} | Cost: {latency_cost}')


## Ex 4: Cost-Aware Routing Math


In [ ]:
print('Cost-per-1M tokens (output, illustrative 2026 rates):')
for model, price in [
    ('gemini-2.5-flash', 0.40),
    ('claude-haiku-4-5', 1.00),
    ('gpt-4o-mini',       0.60),
    ('claude-sonnet-4-6',15.00),
    ('claude-opus-4-7',  75.00),
]:
    print(f'  {model:20s} ${price:6.2f}/M output')
print()
print('Routing split: 70% flash + 22% haiku + 8% sonnet  vs  100% sonnet')
weighted = 0.70*0.40 + 0.22*1.00 + 0.08*15.00
print(f'  routed:  ${weighted:.2f}/M effective')
print(f'  baseline: $15.00/M')
print(f'  savings:  {(1-weighted/15.0)*100:.1f}%')


## Ex 5: Virtual Keys per Team


In [ ]:
print('Virtual key design:')
for env, models, cap, purpose in [
    ('dev',    'flash only',        '$5/day',   'experimentation, blast radius zero'),
    ('staging','haiku + flash',     '$50/day',  'integration testing + eval jobs'),
    ('prod',   'all',                'none',    'customer traffic, budget alerts only'),
    ('ci',     'flash only',        '$10/day',  'CI eval gates, scoped to repo OIDC'),
]:
    print(f'  {env:8s} | models={models:22s} | cap={cap:9s} | {purpose}')


## Ex 6: HA Gateway Topology


In [ ]:
print('HA components (2+ of each):')
for component, role, notes in [
    ('L4 LB (nginx/envoy)','Terminate TCP, health-check replicas','no sticky sessions'),
    ('Gateway replicas','Stateless, handle API calls','scale horizontally on QPS'),
    ('Postgres','Virtual keys, audit log, budgets','primary + read replica'),
    ('Redis','Rate limits + semantic cache','sentinel or cluster mode'),
    ('Prometheus','Metrics store','HA pair with remote_write'),
]:
    print(f'  {component:20s}: {role:38s} | {notes}')


---
## Recap
Gateway = single API + fallback + virtual keys + semantic cache + cost routing + observability.
LiteLLM self-host default; Portkey managed; Bifrost for low latency.
Cost math: 70/22/8 routing cuts bill ~85% vs frontier-only.
HA: 2+ replicas, stateless, Postgres + Redis for state, L4 LB with health checks.
